# When Can You Trust Linear Regression?

This project uses simple simulations to see how OLS behaves when common problems appear in the data.

I focus on four issues:

- small samples
- outliers
- multicollinearity
- nonlinearity

I also compare OLS with Huber and Ridge regression and finish with a small real-data example.


## Abstract

OLS is easy to use, but its results can become less reliable when the data do not fit the model well.

I use Monte Carlo simulations to change one problem at a time and examine coefficient estimates, uncertainty, and prediction error. I then compare OLS with Huber regression for outliers and Ridge regression for multicollinearity.

Finally, I apply a similar diagnostic checklist to the diabetes dataset from scikit-learn.


## Introduction

Linear regression is one of the most common tools in economics and statistics.

The important question is not only how to fit a regression, but also whether the result can be trusted.

This project looks at several common problems and asks how they affect OLS estimates and predictions.


## Research Question

**Main question:**

> When do common data problems make OLS less reliable?

I look at:

1. sample size
2. outliers
3. multicollinearity
4. nonlinearity
5. whether Huber or Ridge can help
6. whether the same ideas appear in real data


## Background on Linear Regression

A simple linear model can be written as

$$y = X\beta + \varepsilon$$

OLS chooses the coefficients that minimize the sum of squared residuals.

When the usual assumptions are reasonable, OLS has useful properties such as unbiasedness and good efficiency.

This project focuses on what happens when some of those conditions become weaker.


## Main OLS Assumptions

| Assumption | Simple meaning |
|---|---|
| Linearity | The relationship is reasonably linear |
| Exogeneity | Predictors are not related to the error |
| No perfect multicollinearity | Predictors are not exact copies of each other |
| Constant variance | Error variance is reasonably stable |
| Independence | Observations/errors are not dependent |
| Normality | Useful for exact small-sample inference |
| Adequate sample size | There is enough information to estimate the model |

I focus on sample size, outliers, multicollinearity, and nonlinearity. Other problems, such as autocorrelation, are outside the main experiments.


## Simulation Framework

I start with a simple data-generating process:

$$y = 5 + 2X_1 + 1.5X_2 + \varepsilon$$

where the error has standard deviation 2.5.

For each experiment, I change one feature of the data and keep the rest the same.

Each condition is repeated 1,000 times.

I mainly look at:

- coefficient bias
- variance
- confidence-interval width and coverage
- prediction RMSE

This gives a controlled way to see how OLS reacts to each problem.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.datasets import load_diabetes
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset

BETA0, BETA1, BETA2 = 5.0, 2.0, 1.5
SIGMA = 2.5

rng_test = np.random.default_rng(2024)
X1_TEST = rng_test.normal(0, 1, 5000)
X2_TEST = rng_test.normal(0, 1, 5000)

def fit_ols(X1, X2, y):
    X = sm.add_constant(np.column_stack([X1, X2]))
    model = sm.OLS(y, X).fit()
    ci = model.conf_int()
    return {
        "coef": model.params,
        "ci_low": ci[:, 0],
        "ci_high": ci[:, 1],
        "resid": model.resid
    }

def oos_rmse(coef, gamma=0):
    true = BETA0 + BETA1 * X1_TEST + BETA2 * X2_TEST + gamma * X1_TEST**2
    pred = coef[0] + coef[1] * X1_TEST + coef[2] * X2_TEST
    return np.sqrt(np.mean((true - pred)**2))

def summarize(df, group):
    result = df.groupby(group)["b1_hat"].agg(["mean", "var"]).reset_index()
    result["bias"] = result["mean"] - BETA1
    result["mse"] = result["bias"]**2 + result["var"]
    return result

def boxplot_by(ax, df, group, value):
    groups = sorted(df[group].unique())
    data = [df.loc[df[group] == g, value] for g in groups]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels(groups)
    return groups


## Experimental Design

| Experiment | What changes |
|---|---|
| Sample size | $n=20$ to $1000$ |
| Outliers | 0% to 30% contaminated observations |
| Outliers: method comparison | OLS vs Huber |
| Multicollinearity | correlation from 0 to 0.99 |
| Multicollinearity: method comparison | OLS vs Ridge |
| Nonlinearity | increasing quadratic term |
| Real data | scikit-learn diabetes dataset |

Each simulation uses 1,000 repetitions.


## Results

The following sections show the results from each experiment. The interpretation is summarized later.


### Experiment 1: Sample Size

I change the number of observations and examine how the OLS estimate and its uncertainty change.


In [ ]:
def run_sample_size_experiment(sizes, reps=1000, seed=1):
    rng = np.random.default_rng(seed)
    rows = []

    for n in sizes:
        for _ in range(reps):
            X1 = rng.normal(0, 1, n)
            X2 = rng.normal(0, 1, n)
            y = BETA0 + BETA1*X1 + BETA2*X2 + rng.normal(0, SIGMA, n)

            fit = fit_ols(X1, X2, y)
            rows.append({
                "n": n,
                "b1_hat": fit["coef"][1],
                "ci_width": fit["ci_high"][1] - fit["ci_low"][1],
                "covers": fit["ci_low"][1] <= BETA1 <= fit["ci_high"][1],
                "rmse": oos_rmse(fit["coef"])
            })

    return pd.DataFrame(rows)

df1 = run_sample_size_experiment([20, 50, 100, 250, 500, 1000])
tbl1 = summarize(df1, "n")

tbl1 = tbl1.merge(
    df1.groupby("n")["covers"].mean().rename("coverage"),
    on="n"
).merge(
    df1.groupby("n")["ci_width"].mean().rename("ci_width"),
    on="n"
).merge(
    df1.groupby("n")["rmse"].mean().rename("rmse"),
    on="n"
)

tbl1.round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

boxplot_by(axes[0], df1, "n", "b1_hat")
axes[0].axhline(BETA1, linestyle="--")
axes[0].set(xlabel="Sample size", ylabel="Estimated beta 1", title="Coefficient estimates")

axes[1].plot(tbl1["n"], tbl1["ci_width"], "o-")
axes[1].set(xlabel="Sample size", ylabel="Mean 95% CI width", title="Confidence intervals")

axes[2].plot(tbl1["n"], tbl1["coverage"], "o-")
axes[2].axhline(0.95, linestyle="--")
axes[2].set(xlabel="Sample size", ylabel="Coverage", title="95% CI coverage")

plt.tight_layout()
plt.show()


### Experiment 2: Outliers

I gradually increase the share of observations with unusually large errors.


In [ ]:
def run_outlier_experiment(pcts, reps=1000, n=200, contam_k=10, seed=2):
    rng = np.random.default_rng(seed)
    rows = []

    for pct in pcts:
        p = pct / 100

        for _ in range(reps):
            X1 = rng.normal(0, 1, n)
            X2 = rng.normal(0, 1, n)

            errors = rng.normal(0, SIGMA, n)
            large_errors = rng.normal(0, contam_k*SIGMA, n)
            errors = np.where(rng.random(n) < p, large_errors, errors)

            y = BETA0 + BETA1*X1 + BETA2*X2 + errors
            fit = fit_ols(X1, X2, y)

            rows.append({
                "pct": pct,
                "b1_hat": fit["coef"][1],
                "rmse": oos_rmse(fit["coef"]),
                "resid_kurtosis": stats.kurtosis(fit["resid"])
            })

    return pd.DataFrame(rows)

df2 = run_outlier_experiment([0, 5, 10, 20, 30])
tbl2 = summarize(df2, "pct").merge(
    df2.groupby("pct")["rmse"].mean().rename("rmse"),
    on="pct"
).merge(
    df2.groupby("pct")["resid_kurtosis"].mean().rename("resid_kurtosis"),
    on="pct"
)

tbl2.round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

boxplot_by(axes[0], df2, "pct", "b1_hat")
axes[0].axhline(BETA1, linestyle="--")
axes[0].set(xlabel="Contaminated observations (%)", ylabel="Estimated beta 1", title="Coefficient estimates")

axes[1].plot(tbl2["pct"], tbl2["rmse"], "o-")
axes[1].set(xlabel="Contamination (%)", ylabel="Prediction RMSE", title="Prediction error")

axes[2].plot(tbl2["pct"], tbl2["resid_kurtosis"], "o-")
axes[2].set(xlabel="Contamination (%)", ylabel="Residual kurtosis", title="Residual tails")

plt.tight_layout()
plt.show()


### Experiment 2b: OLS vs Huber Regression

Huber regression gives less weight to observations with very large residuals. I compare it with OLS under the same simulated outliers.


In [ ]:
def run_outlier_method_comparison(pcts, reps=1000, n=200, contam_k=10, seed=2):
    rng = np.random.default_rng(seed)
    rows = []

    for pct in pcts:
        for _ in range(reps):
            X1 = rng.normal(0, 1, n)
            X2 = rng.normal(0, 1, n)
            errors = rng.normal(0, SIGMA, n)
            large_errors = rng.normal(0, contam_k*SIGMA, n)
            errors = np.where(rng.random(n) < pct/100, large_errors, errors)

            y = BETA0 + BETA1*X1 + BETA2*X2 + errors
            X = np.column_stack([X1, X2])

            ols = fit_ols(X1, X2, y)
            huber = HuberRegressor(epsilon=1.35, max_iter=200).fit(X, y)

            rows.extend([
                {"pct": pct, "method": "OLS", "b1_hat": ols["coef"][1],
                 "rmse": oos_rmse(ols["coef"])},
                {"pct": pct, "method": "Huber", "b1_hat": huber.coef_[0],
                 "rmse": oos_rmse([huber.intercept_, huber.coef_[0], huber.coef_[1]])}
            ])

    return pd.DataFrame(rows)

df2b = run_outlier_method_comparison([0, 5, 10, 20, 30])

tbl2b = df2b.groupby(["pct", "method"])["b1_hat"].agg(
    mean_estimate="mean", variance="var"
).reset_index()

tbl2b["bias"] = tbl2b["mean_estimate"] - BETA1
tbl2b["mse"] = tbl2b["bias"]**2 + tbl2b["variance"]

tbl2b = tbl2b.merge(
    df2b.groupby(["pct", "method"])["rmse"].mean().rename("rmse"),
    on=["pct", "method"]
)

tbl2b.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for method in ["OLS", "Huber"]:
    data = tbl2b[tbl2b["method"] == method]
    axes[0].plot(data["pct"], data["mse"], "o-", label=method)
    axes[1].plot(data["pct"], data["rmse"], "o-", label=method)

axes[0].set(xlabel="Contamination (%)", ylabel="Coefficient MSE", title="Coefficient MSE")
axes[1].set(xlabel="Contamination (%)", ylabel="Prediction RMSE", title="Prediction error")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()


### Experiment 3: Multicollinearity

I increase the correlation between the two predictors and observe how the OLS coefficient becomes less precise.


In [ ]:
def run_multicollinearity_experiment(rhos, reps=1000, n=200, seed=3):
    rng = np.random.default_rng(seed)
    rows = []

    for rho in rhos:
        L = np.linalg.cholesky([[1, rho], [rho, 1]])

        for _ in range(reps):
            X = rng.normal(0, 1, (n, 2)) @ L.T
            X1, X2 = X[:, 0], X[:, 1]
            y = BETA0 + BETA1*X1 + BETA2*X2 + rng.normal(0, SIGMA, n)

            fit = fit_ols(X1, X2, y)
            vif = 1 / (1 - np.corrcoef(X1, X2)[0, 1]**2)

            rows.append({
                "rho": rho,
                "b1_hat": fit["coef"][1],
                "ci_width_b1": fit["ci_high"][1] - fit["ci_low"][1],
                "vif": vif,
                "rmse": oos_rmse(fit["coef"])
            })

    return pd.DataFrame(rows)

df3 = run_multicollinearity_experiment([0, .3, .6, .8, .9, .95, .99])

tbl3 = summarize(df3, "rho").merge(
    df3.groupby("rho")["ci_width_b1"].mean().rename("ci_width"),
    on="rho"
).merge(
    df3.groupby("rho")["vif"].mean().rename("vif"),
    on="rho"
)

tbl3.round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

boxplot_by(axes[0], df3, "rho", "b1_hat")
axes[0].axhline(BETA1, linestyle="--")
axes[0].set(xlabel="Correlation", ylabel="Estimated beta 1", title="Coefficient estimates")

axes[1].plot(tbl3["rho"], tbl3["ci_width"], "o-")
axes[1].set(xlabel="Correlation", ylabel="Mean CI width", title="Confidence intervals")

axes[2].plot(tbl3["rho"], tbl3["vif"], "o-")
axes[2].axhline(10, linestyle="--")
axes[2].set(xlabel="Correlation", ylabel="VIF", title="Variance inflation")

plt.tight_layout()
plt.show()


### Experiment 3b: OLS vs Ridge Regression

Ridge regression adds a penalty that can reduce the instability caused by highly correlated predictors.


In [ ]:
def run_multicollinearity_method_comparison(rhos, reps=1000, n=200, alpha=1.0, seed=3):
    rng = np.random.default_rng(seed)
    rows = []

    for rho in rhos:
        L = np.linalg.cholesky([[1, rho], [rho, 1]])

        for _ in range(reps):
            X = rng.normal(0, 1, (n, 2)) @ L.T
            X1, X2 = X[:, 0], X[:, 1]
            y = BETA0 + BETA1*X1 + BETA2*X2 + rng.normal(0, SIGMA, n)

            ols = fit_ols(X1, X2, y)
            ridge = Ridge(alpha=alpha).fit(X, y)

            rows.extend([
                {"rho": rho, "method": "OLS", "b1_hat": ols["coef"][1],
                 "rmse": oos_rmse(ols["coef"])},
                {"rho": rho, "method": "Ridge", "b1_hat": ridge.coef_[0],
                 "rmse": oos_rmse([ridge.intercept_, ridge.coef_[0], ridge.coef_[1]])}
            ])

    return pd.DataFrame(rows)

df3b = run_multicollinearity_method_comparison([0, .3, .6, .8, .9, .95, .99])

tbl3b = df3b.groupby(["rho", "method"])["b1_hat"].agg(
    mean_estimate="mean", variance="var"
).reset_index()

tbl3b["bias"] = tbl3b["mean_estimate"] - BETA1
tbl3b["mse"] = tbl3b["bias"]**2 + tbl3b["variance"]

tbl3b = tbl3b.merge(
    df3b.groupby(["rho", "method"])["rmse"].mean().rename("rmse"),
    on=["rho", "method"]
)

tbl3b.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for method in ["OLS", "Ridge"]:
    data = tbl3b[tbl3b["method"] == method]
    axes[0].plot(data["rho"], data["mse"], "o-", label=method)
    axes[1].plot(data["rho"], data["rmse"], "o-", label=method)

axes[0].set(xlabel="Correlation", ylabel="Coefficient MSE", title="Coefficient MSE")
axes[1].set(xlabel="Correlation", ylabel="Prediction RMSE", title="Prediction error")

for ax in axes:
    ax.legend()

plt.tight_layout()
plt.show()


### Experiment 4: Nonlinearity

I add a quadratic term to the true relationship but continue fitting a linear model.

The predictor is symmetric around zero, so the linear coefficient can remain centered even though the model misses the curved relationship.


In [ ]:
def run_nonlinearity_experiment(gammas, reps=1000, n=300, seed=4):
    rng = np.random.default_rng(seed)
    rows = []

    for gamma in gammas:
        for _ in range(reps):
            X1 = rng.normal(0, 1, n)
            X2 = rng.normal(0, 1, n)
            y = (
                BETA0 + BETA1*X1 + BETA2*X2
                + gamma*X1**2
                + rng.normal(0, SIGMA, n)
            )

            fit = fit_ols(X1, X2, y)

            rows.append({
                "gamma": gamma,
                "b1_hat": fit["coef"][1],
                "rmse": oos_rmse(fit["coef"], gamma),
                "resid_curv_corr": np.corrcoef(
                    fit["resid"], X1**2
                )[0, 1]
            })

    return pd.DataFrame(rows)

df4 = run_nonlinearity_experiment([0, .5, 1, 2, 4, 8])

tbl4 = summarize(df4, "gamma").merge(
    df4.groupby("gamma")["rmse"].mean().rename("rmse"),
    on="gamma"
).merge(
    df4.groupby("gamma")["resid_curv_corr"].mean().rename("resid_curv_corr"),
    on="gamma"
)

tbl4.round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(tbl4["gamma"], tbl4["rmse"], "o-")
axes[0].set(xlabel="Curvature", ylabel="Prediction RMSE", title="Prediction error")

axes[1].plot(tbl4["gamma"], tbl4["resid_curv_corr"], "o-")
axes[1].set(xlabel="Curvature", ylabel="Residual / curvature correlation",
            title="Residual pattern")

boxplot_by(axes[2], df4, "gamma", "b1_hat")
axes[2].axhline(BETA1, linestyle="--")
axes[2].set(xlabel="Curvature", ylabel="Estimated beta 1", title="Coefficient estimates")

plt.tight_layout()
plt.show()


## Interpretation

### Sample size

A small sample mainly makes estimates less precise. As the sample gets larger, the confidence intervals become narrower.

### Outliers

The symmetric outlier setup does not create much average bias, but it makes individual estimates and predictions much less stable.

### Multicollinearity

Highly correlated predictors make individual coefficients difficult to estimate precisely. The main problem is larger standard errors rather than systematic bias.

### Nonlinearity

A linear model can miss an important relationship even when the main coefficient looks reasonable. Prediction error and residual patterns can reveal the problem.

The main lesson is that different problems affect different parts of a regression result.


## Overall Robustness Assessment

The simulations show that there is no single "OLS problem."

Some problems mainly increase uncertainty, while others mainly hurt prediction.

The useful approach is to diagnose the specific problem before choosing a solution.


In [ ]:
assumptions = ["Sample size", "Outliers", "Multicollinearity", "Nonlinearity"]
dimensions = ["Coefficient\nbias", "Estimator\nvariance", "Prediction\nerror (RMSE)", "Inference\n(CI/coverage)"]
scores = np.array([
    [0, 2, 1, 2],
    [1, 2, 2, 1],
    [0, 2, 0, 2],
    [0, 1, 2, 1],
])
labels = np.array([["Low", "High", "Moderate", "High"],
                    ["Moderate", "High", "High", "Moderate"],
                    ["Low", "High", "Low", "High"],
                    ["Low", "Moderate", "High", "Moderate"]])

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.imshow(scores, cmap="RdYlGn_r", vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(len(dimensions))); ax.set_xticklabels(dimensions, fontsize=9)
ax.set_yticks(range(len(assumptions))); ax.set_yticklabels(assumptions, fontsize=9.5)
for i in range(scores.shape[0]):
    for j in range(scores.shape[1]):
        ax.text(j, i, labels[i, j], ha="center", va="center", fontsize=9, fontweight="bold",
                color="white" if scores[i, j] == 2 else "black")
ax.set_title("Figure 5 — Impact of Each Violation, by Dimension", fontweight="bold", pad=12)
for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout(); plt.show()

| Problem | Main effect | Possible response |
|---|---|---|
| Small sample | Less precision | Collect more data or report uncertainty clearly |
| Outliers | Less stable estimates and predictions | Check observations; consider robust regression |
| Multicollinearity | Large standard errors | Check VIF; combine or remove predictors; consider Ridge |
| Nonlinearity | Poor prediction and residual patterns | Transform variables or use nonlinear terms |


## Real-Data Case Study: Diabetes Dataset

I apply the same basic diagnostic ideas to the scikit-learn diabetes dataset.

The dataset has 442 observations and 10 predictors.

Unlike the simulations, the true relationship is unknown here, so the results should be treated as diagnostics rather than proof that a model is correct or incorrect.


In [ ]:
rd_data = load_diabetes(as_frame=True)
X = rd_data.data
y = rd_data.target

Xc = sm.add_constant(X)
rd_model = sm.OLS(y, Xc).fit()

rd_vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns,
    name="vif"
)

rd_table = pd.DataFrame({
    "coef": rd_model.params[1:],
    "std_err": rd_model.bse[1:],
    "p_value": rd_model.pvalues[1:],
    "vif": rd_vif
})

rd_table.sort_values("vif", ascending=False).round(3)


In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(rd_model.resid, Xc)
reset = linear_reset(rd_model, power=3, use_f=True)
cooks = OLSInfluence(rd_model).cooks_distance[0]

print(f"R-squared: {rd_model.rsquared:.3f}")
print(f"Breusch-Pagan p-value: {bp_p:.4f}")
print(f"RESET p-value: {reset.pvalue:.4f}")
print(f"Maximum Cook's distance: {cooks.max():.4f}")
print(f"Points above 4/n: {(cooks > 4/len(y)).sum()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(rd_model.fittedvalues, rd_model.resid, alpha=0.5)
axes[0].axhline(0, linestyle="--")
axes[0].set(xlabel="Fitted values", ylabel="Residuals", title="Residuals vs fitted")

vif_sorted = rd_vif.sort_values()
axes[1].barh(vif_sorted.index, vif_sorted.values)
axes[1].axvline(10, linestyle="--")
axes[1].set(xlabel="VIF", title="Multicollinearity")

plt.tight_layout()
plt.show()


**What the checklist found**

Some predictors have high VIF values, suggesting multicollinearity.

The residual tests also suggest that the linear model may have problems with variance and functional form.

This example shows why looking only at coefficients and R-squared is not enough.


## Lessons

- Look at residual plots.
- Check VIF when there are several related predictors.
- Treat a small sample as a source of uncertainty.
- Investigate unusual observations instead of automatically deleting them.
- Match the method to the problem.
- Do not assume that an insignificant coefficient automatically means the model is bad.


## Limitations

- The simulations use specific parameter choices.
- Only one simple outlier setup is used.
- Only one type of nonlinearity is tested.
- Real data can have several problems at the same time.
- Huber and Ridge use simple fixed settings rather than extensive tuning.
- The real-data example uses only one dataset.

The exact numerical results should therefore not be treated as universal rules.


## Conclusion

OLS is not simply trustworthy or untrustworthy.

Its reliability depends on the problem in the data and on what we want from the model.

The simulations show that small samples, outliers, multicollinearity, and nonlinearity affect regression results in different ways. Simple diagnostics can help identify these problems, and alternative methods such as Huber or Ridge can sometimes improve the results.

The main lesson is simple: **fit the regression, check the assumptions, understand the problem, and then decide how much confidence to place in the result.**
